## 1º Carregar o Dataset
 **Importação dos Dados:** 
  * Realizar a leitura do arquivo `.csv` que está no projeto na pasta `1_DataSet` o arquivo está como `.zip` utilizando bibliotecas adequadas (ex: `pd.read_csv()` no Pandas).
**Inspeção Inicial:**
  * Verificação das primeiras e últimas linhas (`head()` / `tail()`).
  * Análise preliminar de dimensões (linhas e colunas) e tipos de dados estruturais (`info()`).

## Vamos ler diretamente para o Pandas (Sem descompactar no disco)
Esta é uma boa prática para evitar redundância de arquivos e economizar espaço de armazenamento:

In [25]:
import pandas as pd
import io
import zipfile

In [26]:
DataSet_INEP_zip = '../datasets/Tabela_Escola_2025_INEP.zip'
DataSet_IDEB_zip = '../datasets/IDEB_Escolas_Ensino_medio_2025.zip'

In [27]:
def carregar_dados(DataSet):
    dados = None
    try:
        with zipfile.ZipFile(DataSet, 'r') as zip_ref:
            nome_csv = [f for f in zip_ref.namelist() if f.endswith('.csv')][0]
            with zip_ref.open(nome_csv) as arquivo_csv:
                dados = pd.read_csv(arquivo_csv, sep=None, engine='python', encoding='latin1')
        print(f"Arquivo {DataSet} carregado com sucesso!")

    except Exception as error:
        print(f"Erro ao carregar os dados do arquivo ZIP.{error}")
    return dados

## Vamos inicialmente trabalhar com o DataSet com os dados do INEP, esse DataSet contém dados pertinentes á todas as escolas que participaram do Censo Escolar de 2025.
**Vamos inicialmente carregar o DataSet e realizar uma limpeza de algumas features que não serão consideradas nas etapas seguinte.**
*É Importante nessa etapa o acompanhamento das descrições que compõem cada features, essa descrição está em 01.Entendimento_dos_dados.ipynb


In [28]:
df_INEP = carregar_dados(DataSet_INEP_zip)
df_INEP.info()
print("\n\nResumo estatístico do DataFrame df_INEP:\n")
df_INEP.describe(include='all')


Arquivo ../datasets/Tabela_Escola_2025_INEP.zip carregado com sucesso!
<class 'pandas.DataFrame'>
RangeIndex: 214192 entries, 0 to 214191
Columns: 290 entries, NU_ANO_CENSO to IN_ESP_EXCLUSIVA_PROF
dtypes: float64(268), int64(10), str(12)
memory usage: 473.9 MB


Resumo estatístico do DataFrame df_INEP:



,NU_ANO_CENSO,NO_REGIAO,CO_REGIAO,NO_UF,SG_UF,CO_UF,NO_MUNICIPIO,CO_MUNICIPIO,NO_REGIAO_GEOG_INTERM,CO_REGIAO_GEOG_INTERM,...,IN_ESP_EXCLUSIVA_MEDIO_FIC,IN_ESP_EXCLUSIVA_MEDIO_NORMAL,IN_COMUM_EJA_FUND,IN_COMUM_EJA_MEDIO,IN_COMUM_EJA_PROF,IN_ESP_EXCLUSIVA_EJA_FUND,IN_ESP_EXCLUSIVA_EJA_MEDIO,IN_ESP_EXCLUSIVA_EJA_PROF,IN_COMUM_PROF,IN_ESP_EXCLUSIVA_PROF
count,214192.0,214191,214192.000000,214191,214191,214192.000000,214191,2.141920e+05,214191,214191.000000,...,180540.000000,180540.000000,180540.000000,180540.000000,180540.000000,180540.000000,180540.000000,180540.000000,180540.000000,180540.000000
unique,NaN,5,NaN,27,27,NaN,5298,NaN,133,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,Sudeste,NaN,São Paulo,SP,NaN,São Paulo,NaN,São Paulo,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,74567,NaN,33616,33616,NaN,7905,NaN,15338,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,2025.0,NaN,2.660459,NaN,NaN,30.348944,NaN,3.050211e+06,NaN,3038.258741,...,0.000039,0.000011,0.129052,0.048444,0.010009,0.006735,0.000343,0.000061,0.033184,0.000255
std,0.0,NaN,1.025717,NaN,NaN,9.480302,NaN,9.517342e+05,NaN,948.387129,...,0.006227,0.003328,0.335258,0.214702,0.099543,0.081793,0.018528,0.007805,0.179117,0.015960
min,2025.0,NaN,1.000000,NaN,NaN,11.000000,NaN,1.100015e+06,NaN,1101.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2025.0,NaN,2.000000,NaN,NaN,23.000000,NaN,2.313757e+06,NaN,2306.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2025.0,NaN,3.000000,NaN,NaN,31.000000,NaN,3.119856e+06,NaN,3102.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,2025.0,NaN,3.000000,NaN,NaN,35.000000,NaN,3.548500e+06,NaN,3504.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


## Vamos retirar do DataSet as features que não são necessárias e selecionar somente os dados das escolas públicas, pois a nota do Ideb é uma avaliação somente de escolas públicas.

In [29]:
print("\n\nRemovendo variáveis:\n")

variaveis_para_remover = [
    "NU_ANO_CENSO",
    "NO_REGIAO",
    "CO_REGIAO",
    "NO_UF",
    "NO_MUNICIPIO",
    "CO_MUNICIPIO",
    "NO_REGIAO_GEOG_INTERM",
    "CO_REGIAO_GEOG_INTERM",
    "NO_REGIAO_GEOG_IMED",
    "CO_REGIAO_GEOG_IMED",
    "NO_MESORREGIAO",
    "CO_MESORREGIAO",
    "NO_MICRORREGIAO",
    "CO_MICRORREGIAO",
    "NO_DISTRITO",
    "CO_DISTRITO",
    "NO_SUBDISTRITO",
    "CO_SUBDISTRITO",
    "NO_REGIAO_ADMINISTRATIVA",
    "CO_REGIAO_ADMINISTRATIVA",
    "TP_CATEGORIA_ESCOLA_PRIVADA",
    "TP_LOCALIZACAO_DIFERENCIADA",
    "DS_ENDERECO",
    "NU_ENDERECO",
    "DS_COMPLEMENTO",
    "NO_BAIRRO",
    "CO_CEP",
    "NU_DDD",
    "NU_TELEFONE",
    "LATITUDE",
    "LONGITUDE",
    "TP_SITUACAO_FUNCIONAMENTO",
    "CO_ORGAO_REGIONAL",
    "DT_ANO_LETIVO_INICIO",
    "DT_ANO_LETIVO_TERMINO",
    "IN_VINCULO_SECRETARIA_EDUCACAO",
    "IN_VINCULO_SEGURANCA_PUBLICA",
    "IN_VINCULO_SECRETARIA_SAUDE",
    "IN_VINCULO_OUTRO_ORGAO",
    "IN_PODER_PUBLICO_PARCERIA",
    "TP_PODER_PUBLICO_PARCERIA",
    "IN_CONVENIADA_PP",
    "TP_CONVENIO_PODER_PUBLICO",
    "IN_FORMA_CONT_TERMO_COLABORA",
    "IN_FORMA_CONT_TERMO_FOMENTO",
    "IN_FORMA_CONT_ACORDO_COOP",
    "IN_FORMA_CONT_PRESTACAO_SERV",
    "IN_FORMA_CONT_COOP_TEC_FIN",
    "IN_FORMA_CONT_CONSORCIO_PUB",
    "IN_FORMA_CONT_MU_TERMO_COLAB",
    "IN_FORMA_CONT_MU_TERMO_FOMENTO",
    "IN_FORMA_CONT_MU_ACORDO_COOP",
    "IN_FORMA_CONT_MU_PREST_SERV",
    "IN_FORMA_CONT_MU_COOP_TEC_FIN",
    "IN_FORMA_CONT_MU_CONSORCIO_PUB",
    "IN_FORMA_CONT_ES_TERMO_COLAB",
    "IN_FORMA_CONT_ES_TERMO_FOMENTO",
    "IN_FORMA_CONT_ES_ACORDO_COOP",
    "IN_FORMA_CONT_ES_PREST_SERV",
    "IN_FORMA_CONT_ES_COOP_TEC_FIN",
    "IN_FORMA_CONT_ES_CONSORCIO_PUB",
    "IN_TIPO_ATEND_ESCOLARIZACAO",
    "IN_TIPO_ATEND_AC",
    "IN_TIPO_ATEND_AEE",
    "IN_MANT_ESCOLA_PRIVADA_EMP",
    "IN_MANT_ESCOLA_PRIVADA_ONG",
    "IN_MANT_ESCOLA_PRIVADA_OSCIP",
    "IN_MANT_ESCOLA_PRIV_ONG_OSCIP",
    "IN_MANT_ESCOLA_PRIVADA_SIND",
    "IN_MANT_ESCOLA_PRIVADA_SIST_S",
    "IN_MANT_ESCOLA_PRIVADA_S_FINS",
    "NU_CNPJ_ESCOLA_PRIVADA",
    "NU_CNPJ_MANTENEDORA",
    "TP_REGULAMENTACAO",
    "TP_RESPONSAVEL_REGULAMENTACAO",
    "CO_ESCOLA_SEDE_VINCULADA",
    "CO_IES_OFERTANTE",
    "IN_LOCAL_FUNC_OUTROS"
]

df_INEP_limpo = df_INEP.drop(columns=variaveis_para_remover, errors='ignore')
df_INEP_limpo_rede_publica = df_INEP_limpo[df_INEP_limpo['CO_REDE'] == 1]
df_INEP_limpo_rede_publica.drop_duplicates(inplace=True)

print("\n\nDataFrame df_INEP_limpo_rede_publica criado com sucesso!\n")
df_INEP_limpo_rede_publica.info()




Removendo variáveis:



DataFrame df_INEP_limpo_rede_publica criado com sucesso!

<class 'pandas.DataFrame'>
Index: 162434 entries, 0 to 214190
Columns: 232 entries, SG_UF to IN_ESP_EXCLUSIVA_PROF
dtypes: float64(225), int64(5), str(2)
memory usage: 288.8 MB


In [30]:
df_INEP_limpo_rede_publica.CO_ENTIDADE.info()

<class 'pandas.Series'>
Index: 162434 entries, 0 to 214190
Series name: CO_ENTIDADE
Non-Null Count   Dtype
--------------   -----
162434 non-null  int64
dtypes: int64(1)
memory usage: 2.5 MB


## Trabalhando com dados do resultado do Ideb.

In [31]:
df_IDEB= carregar_dados(DataSet_IDEB_zip)
df_IDEB.info()
print("\n\nResumo estatístico do DataFrame df_IDEB:\n")
df_IDEB.describe(include='all')

Arquivo ../datasets/IDEB_Escolas_Ensino_medio_2025.zip carregado com sucesso!
<class 'pandas.DataFrame'>
RangeIndex: 22171 entries, 0 to 22170
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   Sigla da UF          22171 non-null  str  
 1   Código do Município  22171 non-null  int64
 2   Nome do Município    22171 non-null  str  
 3   Código da Escola     22171 non-null  int64
 4   Nome da Escola       22171 non-null  str  
 5   Rede                 22171 non-null  str  
 6   IDEB_2025            22170 non-null  str  
dtypes: int64(2), str(5)
memory usage: 1.2 MB


Resumo estatístico do DataFrame df_IDEB:



,Sigla da UF,Código do Município,Nome do Município,Código da Escola,Nome da Escola,Rede,IDEB_2025
count,22171,2.217100e+04,22171,2.217100e+04,22171,22171,22170
unique,27,NaN,5291,NaN,21403,4,65
top,SP,NaN,São Paulo,NaN,EE DE ENSINO MEDIO,Estadual,-
freq,4322,NaN,710,NaN,18,20081,4587
mean,NaN,3.236898e+06,NaN,3.234697e+07,NaN,NaN,NaN
std,NaN,9.821404e+05,NaN,9.793113e+06,NaN,NaN,NaN
min,NaN,1.100015e+06,NaN,1.100006e+07,NaN,NaN,NaN
25%,NaN,2.604858e+06,NaN,2.605412e+07,NaN,NaN,NaN
50%,NaN,3.300407e+06,NaN,3.300831e+07,NaN,NaN,NaN
75%,NaN,4.100400e+06,NaN,4.100148e+07,NaN,NaN,NaN


In [32]:
print("\n\nRemovendo dados de escolas privadas e de escolas com IDEB não disponível:\n")

df_filtrado = df_IDEB[(df_IDEB['IDEB_2025'].notna()) & (df_IDEB['IDEB_2025'].str.strip() != '-')]
df_filtrado.drop_duplicates(inplace=True)
df_filtrado.info()



Removendo dados de escolas privadas e de escolas com IDEB não disponível:

<class 'pandas.DataFrame'>
Index: 17583 entries, 1 to 22169
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   Sigla da UF          17583 non-null  str  
 1   Código do Município  17583 non-null  int64
 2   Nome do Município    17583 non-null  str  
 3   Código da Escola     17583 non-null  int64
 4   Nome da Escola       17583 non-null  str  
 5   Rede                 17583 non-null  str  
 6   IDEB_2025            17583 non-null  str  
dtypes: int64(2), str(5)
memory usage: 1.1 MB


In [33]:
print("\n\nRemovendo variáveis, e deixando apenas Código da Escola e nota do IDEB\n")

variaveis_para_remover = [
    "Sigla da UF",
    "Código do Município",
    "Nome do Município",
    "Nome da Escola",
    "Rede"]

df_IBEB = df_filtrado.drop(columns=variaveis_para_remover, errors='ignore')
df_IBEB.info()



Removendo variáveis, e deixando apenas Código da Escola e nota do IDEB

<class 'pandas.DataFrame'>
Index: 17583 entries, 1 to 22169
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Código da Escola  17583 non-null  int64
 1   IDEB_2025         17583 non-null  str  
dtypes: int64(1), str(1)
memory usage: 412.1 KB


## Vamos unir os dados de ambas os dataset em um único dataset onde teremos dados das escolas por meio do Censo escolar e dados da nota da escola no Ibed, vamos usar para isso o ID de identificação da Escola.

In [34]:
print("\n\n Unindo os dois Dataframes usando o Código da Escola como chave de junção:\n")

df_final = pd.merge(df_INEP_limpo_rede_publica, df_IBEB, left_on='CO_ENTIDADE', right_on='Código da Escola', how='inner')
print("\n\nDataFrame df_final criado com sucesso!\n")
df_final.info()
df_final.columns.tolist()



 Unindo os dois Dataframes usando o Código da Escola como chave de junção:



DataFrame df_final criado com sucesso!

<class 'pandas.DataFrame'>
RangeIndex: 17583 entries, 0 to 17582
Columns: 234 entries, SG_UF to IDEB_2025
dtypes: float64(225), int64(6), str(3)
memory usage: 31.4 MB


['SG_UF',
 'CO_UF',
 'NO_ENTIDADE',
 'CO_ENTIDADE',
 'CO_REDE',
 'TP_DEPENDENCIA',
 'TP_LOCALIZACAO',
 'IN_LOCAL_FUNC_PREDIO_ESCOLAR',
 'TP_OCUPACAO_PREDIO_ESCOLAR',
 'IN_LOCAL_FUNC_SOCIOEDUCATIVO',
 'IN_LOCAL_FUNC_UNID_PRISIONAL',
 'IN_LOCAL_FUNC_PRISIONAL_SOCIO',
 'IN_LOCAL_FUNC_GALPAO',
 'TP_OCUPACAO_GALPAO',
 'IN_LOCAL_FUNC_SALAS_OUTRA_ESC',
 'IN_PREDIO_COMPARTILHADO',
 'IN_AGUA_POTAVEL',
 'IN_AGUA_REDE_PUBLICA',
 'IN_AGUA_POCO_ARTESIANO',
 'IN_AGUA_CACIMBA',
 'IN_AGUA_FONTE_RIO',
 'IN_AGUA_INEXISTENTE',
 'IN_AGUA_CARRO_PIPA',
 'IN_ENERGIA_REDE_PUBLICA',
 'IN_ENERGIA_GERADOR_FOSSIL',
 'IN_ENERGIA_RENOVAVEL',
 'IN_ENERGIA_INEXISTENTE',
 'IN_ESGOTO_REDE_PUBLICA',
 'IN_ESGOTO_FOSSA_SEPTICA',
 'IN_ESGOTO_FOSSA_COMUM',
 'IN_ESGOTO_FOSSA',
 'IN_ESGOTO_INEXISTENTE',
 'IN_LIXO_SERVICO_COLETA',
 'IN_LIXO_QUEIMA',
 'IN_LIXO_ENTERRA',
 'IN_LIXO_DESTINO_FINAL_PUBLICO',
 'IN_LIXO_DESCARTA_OUTRA_AREA',
 'IN_TRATAMENTO_LIXO_SEPARACAO',
 'IN_TRATAMENTO_LIXO_REUTILIZA',
 'IN_TRATAMENTO_LIXO_RECICLA

## Limpeza do df_final após o Merge

Após unir os dois datasets, realizamos a limpeza necessária antes de salvar e modelar.

**Tratamentos aplicados:**

| # | Problema | Ação |
|---|---|---|
| 1 | Linhas duplicadas | Remover |
| 2 | Colunas com >50% de NaN | **Manter** (registrar no log para decisão consciente) |
| 3 | Colunas constantes (std = 0) | Remover — não contribuem para o modelo |
| 4 | `Código da Escola` (redundante com `CO_ENTIDADE`) | Remover |
| 5 | NaN residuais em colunas numéricas | Preencher com **0** |


In [35]:
import numpy as np

print("=" * 60)
print("  LIMPEZA DO df_final PÓS-MERGE")
print("=" * 60)
print(f"\n  Shape inicial: {df_final.shape}")

# ── 1. Remover linhas duplicadas ─────────────────────────────────────────────
duplicatas = df_final.duplicated().sum()
df_final.drop_duplicates(inplace=True)
print(f"\n  [1] Duplicatas removidas: {duplicatas}")
print(f"      Shape: {df_final.shape}")

# ── 2. Colunas com >50% NaN — MANTER, apenas registrar ───────────────────────
pct_nan = df_final.isnull().mean()
colunas_muitos_nan = pct_nan[pct_nan > 0.50].index.tolist()
print(f"\n  [2] Colunas com >50% de NaN (mantidas, apenas informativo): {len(colunas_muitos_nan)}")
for col in colunas_muitos_nan:
    print(f"      '{col}': {pct_nan[col]*100:.1f}% nulos")

# ── 3. Remover colunas constantes (std = 0) ───────────────────────────────────
numericas = df_final.select_dtypes(include=[np.number])
colunas_constantes = numericas.columns[numericas.std() == 0].tolist()
df_final.drop(columns=colunas_constantes, inplace=True, errors='ignore')
print(f"\n  [3] Colunas constantes removidas (std=0): {len(colunas_constantes)}")
print(f"      {colunas_constantes}")

# ── 4. Remover coluna de identificação redundante ─────────────────────────────
df_final.drop(columns=['Código da Escola'], inplace=True, errors='ignore')
print(f"\n  [4] Coluna 'Código da Escola' removida (redundante com CO_ENTIDADE)")

# ── 5. Preencher NaN residuais com 0 ─────────────────────────────────────────
nan_antes = df_final.isnull().sum()
colunas_com_nan = nan_antes[nan_antes > 0].index.tolist()
print(f"\n  [5] Colunas com NaN residuais preenchidas com 0: {len(colunas_com_nan)}")
for col in colunas_com_nan:
    print(f"      '{col}': {nan_antes[col]} NaN → preenchidos com 0")
df_final[colunas_com_nan] = df_final[colunas_com_nan].fillna(0)

print(f"\n  Shape final: {df_final.shape}")
print(f"  NaN restantes: {df_final.isnull().sum().sum()}")
print("=" * 60)


  LIMPEZA DO df_final PÓS-MERGE

  Shape inicial: (17583, 234)

  [1] Duplicatas removidas: 0
      Shape: (17583, 234)

  [2] Colunas com >50% de NaN (mantidas, apenas informativo): 11
      'TP_OCUPACAO_GALPAO': 99.6% nulos
      'TP_INDIGENA_LINGUA': 99.5% nulos
      'CO_LINGUA_INDIGENA_1': 99.7% nulos
      'CO_LINGUA_INDIGENA_2': 99.9% nulos
      'CO_LINGUA_INDIGENA_3': 100.0% nulos
      'IN_RESERVA_PPI': 94.1% nulos
      'IN_RESERVA_RENDA': 94.1% nulos
      'IN_RESERVA_PUBLICA': 94.1% nulos
      'IN_RESERVA_PCD': 94.1% nulos
      'IN_RESERVA_OUTROS': 94.1% nulos
      'IN_RESERVA_NENHUMA': 94.1% nulos

  [3] Colunas constantes removidas (std=0): 6
      ['CO_REDE', 'IN_ESCOLARIZACAO', 'IN_REGULAR', 'IN_ESP_EXCLUSIVA_CRECHE', 'IN_ESP_EXCLUSIVA_MEDIO_FIC', 'IN_ESP_EXCLUSIVA_MEDIO_NORMAL']

  [4] Coluna 'Código da Escola' removida (redundante com CO_ENTIDADE)

  [5] Colunas com NaN residuais preenchidas com 0: 13
      'TP_OCUPACAO_PREDIO_ESCOLAR': 90 NaN → preenchidos com 0


## O nosso target que é a nota das escolas na avaliação do Ideb está como dtypes strig e temos que transforma-lo para flot, para que possamos utilizar o modelo de regressão para o nosso modelo.

In [36]:
#Vamos isolar para conferência quais são as fetures que estão como dtypes string
coluna_str = [col for col in df_final.columns if df_final[col].dtype == 'string']
df_final[coluna_str]

,SG_UF,NO_ENTIDADE,IDEB_2025
0,RO,EEEMTI JUSCELINO KUBITSCHEK DE OLIVEIRA,"5,1"
1,RO,COLEGIO TIRADENTES DA POLICIA MILITAR - CTPM XI,"4,8"
2,RO,EEEFM CORA CORALINA,"3,4"
3,RO,EEEFM ANISIO TEIXEIRA,"3,8"
4,RO,COLEGIO TIRADENTES DA POLICIA MILITAR - CTPM III,"4,8"
...,...,...,...
17578,DF,CED ZUMBI DOS PALMARES,"3,4"
17579,DF,CED SAO FRANCISCO,"3,1"
17580,DF,CED JARDINS MANGUEIRAL,"4,3"
17581,DF,CED AGROURBANO IPE RIACHO FUNDO,"3,9"


# Vamos agora atuar na feture IDEB_2025, primeiro transformando o separado "," para "." e em seguida transformando para numeric
# e mostramos as mudanças

In [37]:
df_final["IDEB_2025"] = df_final["IDEB_2025"].str.replace(",", ".")

In [38]:
print("\n\n Alterando o dtype da feature para numeric \n\n")
      
df_final["IDEB_2025"] = pd.to_numeric(df_final["IDEB_2025"], errors ="coerce")
df_final["IDEB_2025"].info()



 Alterando o dtype da feature para numeric 


<class 'pandas.Series'>
RangeIndex: 17583 entries, 0 to 17582
Series name: IDEB_2025
Non-Null Count  Dtype  
--------------  -----  
17583 non-null  float64
dtypes: float64(1)
memory usage: 137.5 KB


In [39]:
# Usando 'r' antes do caminho
caminho = r"../datasets/dados_tratados.csv"

df_final.to_csv(caminho, index=False, sep=";", encoding="utf-8-sig")